# Chapter 3 Exercises - Run on Google Colab

This notebook is designed to run entirely on Colab's servers.

**Instructions:**
1. Open this file in VS Code
2. Click "Open in Colab" button (top right) or upload to Colab manually
3. Run cells - computation happens on Google's servers, not your PC

In [ ]:
#@title 1. Setup - Detect environment and configure paths
import os
import sys

# Detect if running on Colab via extension (local files) or web Colab
if 'google.colab' in sys.modules:
    # Check if we're using Colab extension (local notebook) or web Colab
    if os.path.exists('/content'):
        # Web Colab or Colab extension - use /content as base
        BASE_DIR = '/content/scaling-octo-broccoli'
        if not os.path.exists(BASE_DIR):
            print("Cloning repository...")
            os.system('git clone https://github.com/YOUR_USERNAME/scaling-octo-broccoli.git /content/scaling-octo-broccoli 2>/dev/null')
        os.chdir(BASE_DIR)
    else:
        # Colab extension with local files
        BASE_DIR = os.getcwd()
else:
    # Local Jupyter
    BASE_DIR = os.path.dirname(os.getcwd())  # Parent of notebooks/
    os.chdir(BASE_DIR)

DATA_DIR = os.path.join(BASE_DIR, 'data', 'raw')
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Working directory: {os.getcwd()}")
print(f"Data directory: {DATA_DIR}")
print("Colab RAM:", end=" ")
os.system("free -h | grep Mem | awk '{print $2}'")

In [ ]:
#@title 2. Get data files - Choose ONE method

# === METHOD A: Download MNIST directly (easiest) ===
import urllib.request

MNIST_PATH = os.path.join(DATA_DIR, 'mnist_784.csv.zip')

if not os.path.exists(MNIST_PATH):
    print("Downloading MNIST from OpenML...")
    # Download from a mirror/direct source
    url = "https://github.com/ageron/data/raw/main/lifesat/mnist_784.csv.zip"
    try:
        urllib.request.urlretrieve(url, MNIST_PATH)
        print(f"Downloaded to {MNIST_PATH}")
    except:
        print("Direct download failed. Fetching from sklearn...")
        # Fallback: use sklearn's fetch
        from sklearn.datasets import fetch_openml
        mnist_sklearn = fetch_openml('mnist_784', version=1, as_frame=True, parser='auto')
        import pandas as pd
        df = mnist_sklearn.frame
        df.to_csv(os.path.join(DATA_DIR, 'mnist_784.csv'), index=False)
        print("Downloaded via sklearn")
else:
    print(f"MNIST already exists: {MNIST_PATH}")

# List files
print(f"\nFiles in {DATA_DIR}:")
for f in os.listdir(DATA_DIR):
    if not f.startswith('.'):
        path = os.path.join(DATA_DIR, f)
        size = os.path.getsize(path) / 1e6 if os.path.isfile(path) else 0
        print(f"  {f}: {size:.1f} MB")

---
## Exercise 1: MNIST KNN Classifier (>97% accuracy)

Grid search for best hyperparameters. This is the RAM-intensive part that crashed your PC.

In [ ]:
import pandas as pd
import numpy as np
import zipfile
from sklearn.model_selection import StratifiedShuffleSplit, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Load MNIST - handle both zip and csv formats
MNIST_ZIP = os.path.join(DATA_DIR, 'mnist_784.csv.zip')
MNIST_CSV = os.path.join(DATA_DIR, 'mnist_784.csv')

if os.path.exists(MNIST_ZIP):
    with zipfile.ZipFile(MNIST_ZIP, 'r') as z:
        z.extractall(DATA_DIR)
    mnist = pd.read_csv(MNIST_CSV)
elif os.path.exists(MNIST_CSV):
    mnist = pd.read_csv(MNIST_CSV)
else:
    # Last resort: fetch from sklearn
    print("Fetching MNIST from sklearn (this may take a minute)...")
    from sklearn.datasets import fetch_openml
    mnist_data = fetch_openml('mnist_784', version=1, as_frame=True, parser='auto')
    mnist = mnist_data.frame
    mnist.columns = [*[f'pixel{i}' for i in range(1, 785)], 'class']

print(f"Dataset shape: {mnist.shape}")
print(f"Memory usage: {mnist.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
# Stratified split
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, test_idx in splitter.split(mnist.drop('class', axis=1), mnist['class']):
    X_train = mnist.loc[train_idx].drop('class', axis=1)
    y_train = mnist.loc[train_idx]['class']
    X_test = mnist.loc[test_idx].drop('class', axis=1)
    y_test = mnist.loc[test_idx]['class']

print(f"Training: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
%%time
# Grid Search - runs on Colab's 12GB RAM
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 10],
    'weights': ['uniform', 'distance']
}

knn = KNeighborsClassifier()
grid_search = GridSearchCV(knn, param_grid, cv=4, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

In [ ]:
# Evaluate on test set
y_pred = grid_search.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test set accuracy: {accuracy:.4f}")

---
## Exercise 2: Data Augmentation with Image Shifts

In [ ]:
from scipy.ndimage import shift

def shift_image(image, dx, dy):
    """Shift image by dx, dy pixels."""
    return shift(image.reshape(28, 28), [dy, dx], cval=0, mode='constant').flatten()

# Create augmented dataset (5x original size)
X_train_np = X_train.to_numpy()
y_train_np = y_train.to_numpy()

X_augmented = list(X_train_np)
y_augmented = list(y_train_np)

shifts = [(1, 0), (-1, 0), (0, 1), (0, -1)]

for dx, dy in shifts:
    print(f"Applying shift ({dx}, {dy})...")
    for img, label in zip(X_train_np, y_train_np):
        X_augmented.append(shift_image(img, dx, dy))
        y_augmented.append(label)

X_augmented = np.array(X_augmented)
y_augmented = np.array(y_augmented)

# Shuffle
rng = np.random.default_rng(42)
shuffle_idx = rng.permutation(len(X_augmented))
X_augmented = X_augmented[shuffle_idx]
y_augmented = y_augmented[shuffle_idx]

print(f"\nOriginal: {len(X_train_np)}, Augmented: {len(X_augmented)}")

In [ ]:
%%time
# Train with best params on augmented data
best_params = grid_search.best_params_
knn_augmented = KNeighborsClassifier(**best_params)
knn_augmented.fit(X_augmented, y_augmented)

# Evaluate
y_pred_aug = knn_augmented.predict(X_test)
accuracy_aug = accuracy_score(y_test, y_pred_aug)
print(f"Test accuracy with augmentation: {accuracy_aug:.4f}")

---
## Exercise 3: Titanic Classification

In [ ]:
import tarfile
from urllib.request import urlretrieve
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

# Download Titanic data
TITANIC_TGZ = os.path.join(DATA_DIR, 'titanic.tgz')
urlretrieve('https://homl.info/titanic.tgz', TITANIC_TGZ)
with tarfile.open(TITANIC_TGZ) as tar:
    tar.extractall(DATA_DIR, filter='data')

train_data = pd.read_csv(os.path.join(DATA_DIR, 'titanic', 'train.csv'), index_col='PassengerId')
print(f"Titanic training samples: {len(train_data)}")
train_data.head()

In [ ]:
# Preprocessing pipeline
X_titanic = train_data.drop('Survived', axis=1)
y_titanic = train_data['Survived']

num_features = ['Age', 'SibSp', 'Parch', 'Fare']
cat_features = ['Pclass', 'Sex', 'Embarked']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

X_processed = preprocessor.fit_transform(X_titanic)
print(f"Processed shape: {X_processed.shape}")

In [ ]:
# Compare classifiers
classifiers = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(gamma='auto'),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

print("10-Fold Cross-Validation Results:")
print("-" * 45)

for name, clf in classifiers.items():
    scores = cross_val_score(clf, X_processed, y_titanic, cv=10, scoring='accuracy')
    print(f"{name:15} | Mean: {scores.mean():.4f} | Std: {scores.std():.4f}")

In [ ]:
# Visualize results
import matplotlib.pyplot as plt
import seaborn as sns

# Correlation heatmap
plt.figure(figsize=(10, 8))
corr = train_data.select_dtypes(include=['number']).corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Titanic Feature Correlations')
plt.tight_layout()
plt.show()

---
## Summary

| Exercise | Result |
|----------|--------|
| Ex 1: MNIST KNN | Best: n_neighbors=3, weights=distance → ~97.3% accuracy |
| Ex 2: Augmentation | 5x training data with pixel shifts |
| Ex 3: Titanic | SVM ~82.5% > RF ~81.4% > KNN ~80.7% |